# Qwen-Image-Edit-2509：從定妝照與場景圖畫一張首格（Colab）

給 `minimax-h3-colab` skill 的 `images.py` 在 Colab 遠端 GPU 上跑：每個工作上傳 1–3 張參照圖（角色定妝照、場景圖）與一個提示詞檔，用 ComfyUI 原生的 Qwen-Image-Edit-2509（Apache 2.0）畫出一張指定尺寸的首格 PNG，再由 CLI 下載。跟同一個 session 裡的 MiniMax H3 筆記本共用 `/content/ComfyUI`，所以一集只裝一次。

- 模型：`Comfy-Org/Qwen-Image-Edit` 的 `qwen_image_edit_2509_fp8_e4m3fn`（19 GiB）、`Comfy-Org/Qwen-Image` 的 `qwen_2.5_vl_7b_fp8_scaled` 文字編碼器（8.7 GiB）與 `qwen_image_vae`、`lightx2v/Qwen-Image-Lightning` 的 Edit-2509 **8 步** Lightning LoRA（0.8 GiB）。
- 參照圖透過 `TextEncodeQwenImageEditPlus` 的 `image1`–`image3` 進模型（官方 2509 範本的接法）；畫布預設 **1312×736**（跟 H3 首格鏡的畫布一樣，不用再裁），latent 預設是空白 latent（`QIE_LATENT=empty`），也可以從第一張參照圖編碼（`image1`）。
- 所有設定都從環境變數讀（`QIE_*`），由 `images.py` 透過 `colab exec --env` 傳入；網頁模式下也能手動改。

In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), "請在 Colab「執行階段 > 變更執行階段類型」選擇 GPU。"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch CUDA:", torch.version.cuda)
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
# 兩組模型約 29 GiB；跟 H3 共用 session 時磁碟要同時放得下（H3 另要 40 GiB 以上）。
MINIMUM_FREE_DISK = int(os.environ.get("QIE_MIN_FREE_DISK_GIB", "35")) * 2**30
assert shutil.disk_usage("/content").free >= MINIMUM_FREE_DISK, (
    f"Qwen-Image-Edit 至少需要 {MINIMUM_FREE_DISK / 2**30:.0f} GiB 可用磁碟（QIE_MIN_FREE_DISK_GIB 可調）。"
)
COMFY = Path("/content/ComfyUI")
SETUP_MARKER = Path("/content/.qie_comfyui_setup_v1")
if SETUP_MARKER.exists():
    assert COMFY.is_dir(), "Colab setup marker exists, but ComfyUI files are missing; remove /content/.qie_comfyui_setup_v1 and retry."
    print("重用本次 Colab session 已安裝的 ComfyUI 環境")
else:
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "huggingface_hub", "hf_xet", "pillow", "requests"], check=True)
    SETUP_MARKER.touch()
print("ComfyUI:", COMFY)

## 1. 下載模型

檔案存在就跳過；跟 H3 筆記本一樣用 `huggingface_hub`。LoRA 預設是 8 步版（`QIE_LORA=off` 時不掛，改走 20 步、CFG 4 的 base 設定）。

In [ ]:
from huggingface_hub import hf_hub_download

DIFFUSION_REPO = os.environ.get("QIE_DIFFUSION_REPO", "Comfy-Org/Qwen-Image-Edit")
DIFFUSION_REMOTE = os.environ.get("QIE_DIFFUSION_FILE", "split_files/diffusion_models/qwen_image_edit_2509_fp8_e4m3fn.safetensors")
TEXT_REPO = os.environ.get("QIE_TEXT_REPO", "Comfy-Org/Qwen-Image")
TEXT_REMOTE = os.environ.get("QIE_TEXT_FILE", "split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors")
VAE_REPO = TEXT_REPO
VAE_REMOTE = os.environ.get("QIE_VAE_FILE", "split_files/vae/qwen_image_vae.safetensors")
LORA_REPO = os.environ.get("QIE_LORA_REPO", "lightx2v/Qwen-Image-Lightning")
LORA_REMOTE = os.environ.get("QIE_LORA_FILE", "Qwen-Image-Edit-2509/Qwen-Image-Edit-2509-Lightning-8steps-V1.0-bf16.safetensors")
USE_LORA = os.environ.get("QIE_LORA", "on").strip().lower() not in {"off", "0", "false", "no"}


def fetch(repo: str, remote: str, folder: str) -> str:
    target_dir = COMFY / "models" / folder
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / Path(remote).name
    if target.is_file() and target.stat().st_size > 0:
        print("已存在：", target.name)
        return target.name
    downloaded = Path(hf_hub_download(repo_id=repo, filename=remote))
    shutil.copy2(downloaded, target)
    print("下載：", repo, remote, f"→ {target} ({target.stat().st_size / 2**30:.1f} GiB)")
    return target.name


DIFFUSION = fetch(DIFFUSION_REPO, DIFFUSION_REMOTE, "diffusion_models")
TEXT_ENCODER = fetch(TEXT_REPO, TEXT_REMOTE, "text_encoders")
VAE = fetch(VAE_REPO, VAE_REMOTE, "vae")
LORA = fetch(LORA_REPO, LORA_REMOTE, "loras") if USE_LORA else None
print("LoRA:", LORA or "off")

## 2. 設定、提示詞與參照圖

| 環境變數 | 預設 | 用途 |
| --- | --- | --- |
| `QIE_WIDTH`, `QIE_HEIGHT` | 1312, 736 | 首格畫布；兩者要是 16 的倍數，面積不超過 1344×768（跟 H3 一樣，首格不用再裁） |
| `QIE_STEPS`, `QIE_CFG` | LoRA 開：8, 1.0；關：20, 4.0 | 取樣步數與 CFG（官方 2509 範本兩組設定） |
| `QIE_SAMPLER`, `QIE_SCHEDULER`, `QIE_SHIFT` | euler, simple, 3.0 | 官方範本的取樣器與 `ModelSamplingAuraFlow` 的 shift |
| `QIE_SEED` | 隨機 | 重做同一鏡要比較時固定 |
| `QIE_PROMPT_FILE` | — | UTF-8 提示詞檔（CLI 上傳）；沒有就用 `PROMPT_OVERRIDE` |
| `QIE_NEGATIVE_FILE` | — | 反向提示詞檔；沒有就用 `NEGATIVE_OVERRIDE`（預設空字串，Lightning 建議空） |
| `QIE_REFERENCE_IMAGES` | — | JSON 清單，1–3 張（CLI 上傳的遠端路徑）；提示詞裡用「image 1」「image 2」指它們 |
| `QIE_LATENT` | empty | `empty`：空白 latent 畫新構圖；`image1`：從第一張參照圖編碼（適合只改一點的重畫） |
| `QIE_OUTPUT_PATH` | /content/qie_output.png | CLI 下載的檔案 |
| `QIE_JOB_TIMEOUT_SECONDS` | 1800 | 等待推論的上限 |

In [ ]:
import json
from PIL import Image

WIDTH = int(os.environ.get("QIE_WIDTH", "1312"))
HEIGHT = int(os.environ.get("QIE_HEIGHT", "736"))
assert WIDTH % 16 == 0 and HEIGHT % 16 == 0 and WIDTH > 0 and HEIGHT > 0, "QIE_WIDTH 與 QIE_HEIGHT 要是 16 的倍數。"
assert WIDTH * HEIGHT <= 1344 * 768, "面積不可超過 1344×768（首格要跟 H3 畫布一樣大）。"
STEPS = int(os.environ.get("QIE_STEPS", "8" if USE_LORA else "20"))
CFG = float(os.environ.get("QIE_CFG", "1.0" if USE_LORA else "4.0"))
SAMPLER_NAME = os.environ.get("QIE_SAMPLER", "euler")
SCHEDULER_NAME = os.environ.get("QIE_SCHEDULER", "simple")
SHIFT = float(os.environ.get("QIE_SHIFT", "3.0"))
LORA_STRENGTH = float(os.environ.get("QIE_LORA_STRENGTH", "1.0"))
seed_env = os.environ.get("QIE_SEED", "").strip()
SEED = int(seed_env) if seed_env else int.from_bytes(os.urandom(8), "big") % (2**53)
LATENT_SOURCE = os.environ.get("QIE_LATENT", "empty").strip().lower()
assert LATENT_SOURCE in {"empty", "image1"}, "QIE_LATENT 請用 empty 或 image1。"
OUTPUT_PREFIX = os.environ.get("QIE_OUTPUT_PREFIX", "qie")
PROMPT_OVERRIDE = ""   # 網頁模式：在這裡貼提示詞
NEGATIVE_OVERRIDE = ""

prompt_file = os.environ.get("QIE_PROMPT_FILE")
if prompt_file:
    prompt_path = Path(prompt_file)
    assert prompt_path.is_file(), f"Colab CLI 上傳的提示詞檔不存在：{prompt_path}"
    PROMPT = prompt_path.read_text(encoding="utf-8").strip()
    print("使用 CLI 上傳的提示詞：", prompt_path)
else:
    PROMPT = PROMPT_OVERRIDE.strip()
assert PROMPT, "提示詞是空的。"
negative_file = os.environ.get("QIE_NEGATIVE_FILE")
NEGATIVE = Path(negative_file).read_text(encoding="utf-8").strip() if negative_file else NEGATIVE_OVERRIDE.strip()

CLI_MODE = False
REFERENCE_IMAGE_NAMES = []
configured = os.environ.get("QIE_REFERENCE_IMAGES")
if configured:
    source_paths = json.loads(configured)
    assert isinstance(source_paths, list) and 1 <= len(source_paths) <= 3, "請提供 1–3 張參照圖（TextEncodeQwenImageEditPlus 收 image1–image3）。"
    sources = [(Path(p).name, Path(p)) for p in source_paths]
    for _, path in sources:
        assert path.is_file(), f"參照圖不存在：{path}"
    CLI_MODE = True
else:
    from google.colab import files
    uploaded = files.upload()
    assert 1 <= len(uploaded) <= 3, "請上傳 1–3 張參照圖。"
    from io import BytesIO
    sources = [(name, BytesIO(data)) for name, data in uploaded.items()]

input_dir = COMFY / "input"
input_dir.mkdir(parents=True, exist_ok=True)
for number, (source_name, source) in enumerate(sources, start=1):
    with Image.open(source) as opened:
        image = opened.convert("RGB")
    target = input_dir / f"qie_reference_{number:02d}.png"
    image.save(target)
    REFERENCE_IMAGE_NAMES.append(target.name)
    print(f"image {number}：{source_name} → {target} ({image.size[0]}×{image.size[1]})")

print("畫布", WIDTH, "×", HEIGHT, "| steps", STEPS, "| cfg", CFG, "| seed", SEED, "| latent", LATENT_SOURCE, "| LoRA", LORA or "off")
print("PROMPT 長度", len(PROMPT), "字元；NEGATIVE", repr(NEGATIVE[:60]))

## 3. 啟動 ComfyUI 並送出

ComfyUI 已經在跑（例如 H3 筆記本開的）就直接用。節點名稱照官方 2509 範本：`TextEncodeQwenImageEditPlus`、`ModelSamplingAuraFlow`、`CFGNorm`、`KSampler`。

In [ ]:
import time, requests, uuid

PORT = 8188
BASE_URL = f"http://127.0.0.1:{PORT}"
LOG_PATH = Path("/content/comfyui_qie.log")
LOW_VRAM_MODE = os.environ.get("QIE_LOW_VRAM", "auto")
try:
    requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
    print("使用已啟動的 ComfyUI")
except requests.RequestException:
    LOG_FILE = open(LOG_PATH, "a")
    launch = [sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", str(PORT), "--disable-auto-launch"]
    if LOW_VRAM_MODE == "on" or (LOW_VRAM_MODE == "auto" and torch.cuda.get_device_properties(0).total_memory < 48 * 2**30):
        launch.append("--lowvram")
    COMFY_PROCESS = subprocess.Popen(launch, cwd=COMFY, stdout=LOG_FILE, stderr=subprocess.STDOUT)
    for _ in range(180):
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError(f"ComfyUI 啟動失敗；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
        try:
            requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise TimeoutError(f"ComfyUI 啟動逾時；查看 {LOG_PATH}")

object_info = requests.get(BASE_URL + "/object_info", timeout=30).json()
required_nodes = {"UNETLoader", "CLIPLoader", "VAELoader", "LoraLoaderModelOnly", "LoadImage",
                  "TextEncodeQwenImageEditPlus", "ModelSamplingAuraFlow", "CFGNorm", "KSampler",
                  "EmptySD3LatentImage", "ImageScaleToTotalPixels", "VAEEncode", "VAEDecode", "SaveImage"}
missing = required_nodes - object_info.keys()
assert not missing, f"ComfyUI 缺少節點：{sorted(missing)}（ComfyUI 太舊？）。請查看 {LOG_PATH}。"

graph = {
    "1": {"class_type": "UNETLoader", "inputs": {"unet_name": DIFFUSION, "weight_dtype": "default"}},
    "3": {"class_type": "CLIPLoader", "inputs": {"clip_name": TEXT_ENCODER, "type": "qwen_image", "device": "default"}},
    "4": {"class_type": "VAELoader", "inputs": {"vae_name": VAE}},
}
model_ref = ["1", 0]
if LORA:
    graph["2"] = {"class_type": "LoraLoaderModelOnly", "inputs": {"model": model_ref, "lora_name": LORA, "strength_model": LORA_STRENGTH}}
    model_ref = ["2", 0]
graph["5"] = {"class_type": "ModelSamplingAuraFlow", "inputs": {"model": model_ref, "shift": SHIFT}}
graph["6"] = {"class_type": "CFGNorm", "inputs": {"model": ["5", 0], "strength": 1.0}}
encode_inputs = {"clip": ["3", 0], "vae": ["4", 0]}
for index, name in enumerate(REFERENCE_IMAGE_NAMES, start=1):
    node_id = str(10 + index)
    graph[node_id] = {"class_type": "LoadImage", "inputs": {"image": name}}
    encode_inputs[f"image{index}"] = [node_id, 0]
graph["20"] = {"class_type": "TextEncodeQwenImageEditPlus", "inputs": {"prompt": PROMPT, **encode_inputs}}
graph["21"] = {"class_type": "TextEncodeQwenImageEditPlus", "inputs": {"prompt": NEGATIVE, **encode_inputs}}
if LATENT_SOURCE == "empty":
    graph["30"] = {"class_type": "EmptySD3LatentImage", "inputs": {"width": WIDTH, "height": HEIGHT, "batch_size": 1}}
else:
    graph["29"] = {"class_type": "ImageScaleToTotalPixels", "inputs": {"image": ["11", 0], "upscale_method": "lanczos", "megapixels": round(WIDTH * HEIGHT / 1e6, 3)}}
    graph["30"] = {"class_type": "VAEEncode", "inputs": {"pixels": ["29", 0], "vae": ["4", 0]}}
graph["40"] = {"class_type": "KSampler", "inputs": {
    "model": ["6", 0], "seed": SEED, "steps": STEPS, "cfg": CFG, "sampler_name": SAMPLER_NAME,
    "scheduler": SCHEDULER_NAME, "positive": ["20", 0], "negative": ["21", 0], "latent_image": ["30", 0], "denoise": 1.0}}
graph["41"] = {"class_type": "VAEDecode", "inputs": {"samples": ["40", 0], "vae": ["4", 0]}}
graph["42"] = {"class_type": "SaveImage", "inputs": {"images": ["41", 0], "filename_prefix": OUTPUT_PREFIX}}

client_id = uuid.uuid4().hex
response = requests.post(BASE_URL + "/prompt", json={"prompt": graph, "client_id": client_id}, timeout=60)
if response.status_code != 200:
    raise RuntimeError("ComfyUI 拒絕工作：" + response.text[:4000])
prompt_id = response.json()["prompt_id"]
print("已送出，prompt_id =", prompt_id)

## 4. 等待並存檔

輸出存到 `QIE_OUTPUT_PATH`（CLI 模式）或顯示下載（網頁模式）。輸出尺寸一定等於畫布；不等就是圖被節點改過，直接報錯。

In [ ]:
job_timeout_seconds = float(os.environ.get("QIE_JOB_TIMEOUT_SECONDS", "1800"))
deadline = time.monotonic() + job_timeout_seconds
last_report = 0
while time.monotonic() < deadline:
    if 'COMFY_PROCESS' in globals() and COMFY_PROCESS.poll() is not None:
        raise RuntimeError(f"ComfyUI 已退出；查看 {LOG_PATH}\n" + LOG_PATH.read_text()[-5000:])
    history = requests.get(BASE_URL + f"/history/{prompt_id}", timeout=30).json()
    if prompt_id in history:
        run = history[prompt_id]
        if run.get("status", {}).get("status_str") == "error":
            raise RuntimeError("推論失敗：" + json.dumps(run.get("status", {}), ensure_ascii=False, indent=2) + f"\n日誌：{LOG_PATH}")
        outputs = run.get("outputs", {})
        if "42" in outputs:
            break
        if run.get("status", {}).get("status_str") == "success":
            raise RuntimeError("任務已完成但沒有圖片輸出：" + json.dumps(outputs, ensure_ascii=False))
    if time.monotonic() - last_report >= 60:
        print("仍在生成首格…")
        last_report = time.monotonic()
    time.sleep(5)
else:
    raise TimeoutError(f"等待推論超過 {job_timeout_seconds:.0f} 秒；可查詢 /history/{prompt_id}，日誌：{LOG_PATH}")

images = outputs["42"].get("images", [])
assert images, "SaveImage 沒有回傳檔案；請查看 " + str(LOG_PATH)
info = images[0]
image_path = COMFY / "output" / info.get("subfolder", "") / info["filename"]
assert image_path.is_file() and image_path.stat().st_size > 0, f"輸出檔不存在：{image_path}"
with Image.open(image_path) as made:
    assert made.size == (WIDTH, HEIGHT), f"輸出 {made.size} 不等於畫布 {(WIDTH, HEIGHT)}"
print("完成：", image_path, f"({image_path.stat().st_size / 2**20:.2f} MiB, {WIDTH}×{HEIGHT}, seed {SEED})")

if CLI_MODE:
    CLI_OUTPUT_PATH = Path(os.environ.get("QIE_OUTPUT_PATH", "/content/qie_output.png"))
    CLI_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_path, CLI_OUTPUT_PATH)
    print("COLAB_CLI_OUTPUT_PATH:", CLI_OUTPUT_PATH)
    print("QIE_SEED_USED:", SEED)
else:
    from IPython.display import Image as IPImage, display
    from google.colab import files
    display(IPImage(str(image_path)))
    files.download(str(image_path))